# MSOX — Hedged Short LETF Strategy

EGARCH-timed short position in MSOX (2x MSOS), hedged with an LP-optimized
basket of call options spread across maturity buckets.

All strategy code lives in `src/`. This notebook only configures, runs, and
plots — so the same engine runs every ticker.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_all
from src.strategy import fit_egarch_signals, run_final_strategy_v21

TICKER = "MSOX"
print("imports OK")


## 1. Load data

Prices from yfinance, options from the local CSV. The implied-vol solve is the
slow step — expect a few minutes.


In [ ]:
data = load_all(TICKER)
data["df_pricing"].tail()


## 2. Fit the EGARCH regime signal

`fit_window=504` (~2 years) is the baseline used for MSOX.


In [ ]:
data = fit_egarch_signals(data, fit_window=504, percentile_window=252)
data["trade_state"]["action"].value_counts()


## 3. Hedge ratio sweep

`hedge_ratio=0.0` is the naked short — the benchmark to beat.


In [ ]:
rows = []
for hr in [0.0, 0.25, 0.5, 0.75, 1.0]:
    metrics, pnl, state, _ = run_final_strategy_v21(
        data, hedge_ratio=hr, dvt=16000, restrike_band=0.25)
    rows.append({"Hedge ratio": hr, **metrics})

results = pd.DataFrame(rows)
results


## 4. CVaR-optimal hedge ratio

CVaR at 95% — the mean of the worst 5% of episode P&Ls.


In [ ]:
ALPHA = 0.95

cvar_rows = []
for hr in [0.0, 0.25, 0.5, 0.75, 1.0]:
    _, pnl, _, _ = run_final_strategy_v21(
        data, hedge_ratio=hr, dvt=16000, restrike_band=0.25)
    live = pnl[pnl != 0]
    if len(live) == 0:
        continue
    var = np.percentile(live, (1 - ALPHA) * 100)
    cvar = live[live <= var].mean()
    cvar_rows.append({"Hedge ratio": hr,
                      "VaR 95%": round(var, 0),
                      "CVaR 95%": round(cvar, 0),
                      "Total P&L": round(live.sum(), 0)})

cvar_df = pd.DataFrame(cvar_rows)
best = cvar_df.loc[cvar_df["CVaR 95%"].idxmax(), "Hedge ratio"]
print(f"CVaR-optimal hedge ratio: {best}")
cvar_df


## 5. Equity curve


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for hr, label in [(0.0, "Naked short"), (0.5, "Hedged h=0.5"), (1.0, "Hedged h=1.0")]:
    _, pnl, _, _ = run_final_strategy_v21(
        data, hedge_ratio=hr, dvt=16000, restrike_band=0.25)
    ax.plot(pnl.index, 25_000 + pnl.cumsum(), label=label, linewidth=1.6)

ax.axhline(25_000, color="grey", linestyle="--", linewidth=0.8)
ax.set_title(f"{TICKER} — equity curve by hedge ratio")
ax.set_ylabel("Account equity ($)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"../results/{TICKER}/equity_curve.png", dpi=150)
plt.show()
